# Experiment 9: CatBoost, and whether the blending plan is viable

Tuning is closed. LightGBM sits at CV 0.963275 and LB 0.964780, and the leaders are at
0.97102. The forensics say the generator produced a smooth calibrated field with no
leak and no duplicate rows, and the top public notebooks are large stacks. That points
the remaining 0.0063 at ensembling rather than at feature discovery.

CatBoost is the first diverse model. It handles the three categoricals with ordered
target statistics rather than LightGBM's split-based approach, which is a genuinely
different mechanism and the usual reason the two blend well.

## Two things changed since the first version of this notebook

**The decision statistic changed.** The previous version gated the ensembling plan on
the Spearman rank correlation between CatBoost and LightGBM out-of-fold predictions,
with thresholds at 0.97 and 0.99 fixed before the number existed.
`07_oof_diversity.ipynb` measured what those thresholds mean on this data and found
both of them wrong. Within-family correlation across seven LightGBM runs spans 0.9741
to 0.9981, so the 0.99 "same model wearing different hats" bar sits inside the range
one family produces on its own and cannot separate a family from itself. Worse, the
correlation ran the wrong way: the best within-family blend was the *most* correlated
pair, and every less correlated pair scored below the best single model.

So this notebook gates on the **measured rank-blend AUC against the best single
model**, which is the thing we actually want to know and costs nothing extra to read.
Correlation is still printed, as description.

**The structure changed.** The previous version called a full five-fold pipeline twice,
at 2000 iterations, with `verbose=0`, before printing anything. It was estimated at 30
to 50 minutes, ran for two hours with no output, and was killed. That was not the
model's doing. The notebook committed to the full cost before measuring any of it.

This version runs in four stages, cheapest first, and each one prints a projection for
the next:

| stage | what it costs | what it buys |
|---|---|---|
| 1 sizing | one fold, 200 iterations | seconds per 100 iterations, and a projected total |
| 2 determinism | two more of the same | whether CatBoost is reproducible at these settings |
| 3 decision | one fold, full iterations | the blend gain, on 138,000 held-out rows |
| 4 full CV | five folds, full iterations | the ledger row and the submission |

**Stage 4 is off by default.** `RUN_FULL = False` below. Stages 1 to 3 answer the
question, and stage 4 spends the compute only once the answer says it is worth it. To
produce a ledger row, set the flag and Save & Run All top to bottom, per the repo rule
that every ledger number comes from a clean-kernel run.

**No assertions anywhere.** `nbconvert` discards every output in a run where a cell
raises, which already cost this repo one 21-minute run with nothing to show for it.
Checks here set a flag and print loudly, and later stages read the flag.

In [ ]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# THE GATE. Stages 1 to 3 are the probe and are cheap. Stage 4 is the long run.
RUN_FULL = False

# Matches the LightGBM working baseline: lr 0.05, budget lr * iterations = 100.
LR = 0.05
FULL_ITERS = 2000
BENCH_ITERS = 200      # sizing and determinism run here, not at FULL_ITERS
PROBE_FOLD = 0

# 8 logical cores. The Bellwether ingest job holds about one and must not be starved,
# and one is left for the OS. Fixed rather than -1, because a varying thread count is
# a reproducibility hazard in gradient summation. Stage 2 tests that cheaply.
THREADS = 6

# The LightGBM runs CatBoost is compared and blended against.
LGB_REF = "lgbm_lr005_n2000_seed42.npy"     # exp7, same lr and budget, like-for-like
LGB_BEST = "lgbm_lr003_n3333_seed42.npy"    # exp8, the best single model overall

print("catboost", cb.__version__)
print(f"probe: fold {PROBE_FOLD}, bench {BENCH_ITERS} iters, full {FULL_ITERS} iters, "
      f"{THREADS} threads")
print(f"RUN_FULL = {RUN_FULL}" + ("" if RUN_FULL else "   (stage 4 will be skipped)"))

In [ ]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
y = train[TARGET].to_numpy()

# Leak checklist, re-run here rather than assumed. id is a contiguous row index that
# separates train from test perfectly, so it is a guaranteed leak if it reaches the
# model. Printed rather than asserted, for the nbconvert reason above.
checks = {
    "id is not a feature": ID not in FEATURES,
    "target is not a feature": TARGET not in FEATURES,
    "train and test ids do not overlap": not (set(train[ID]) & set(test[ID])),
    "train and test feature lists match":
        list(FEATURES) == [c for c in test.columns if c != ID],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# CatBoost needs categoricals as strings with no NaN, so missing becomes its own level.
# That is not an imputation choice: it keeps missingness as information rather than
# guessing a value, which matches what LightGBM's native routing already does.
Xtr, Xte = train[FEATURES].copy(), test[FEATURES].copy()
for c in CAT_COLS:
    Xtr[c] = Xtr[c].astype("object").fillna("__NA__").astype(str)
    Xte[c] = Xte[c].astype("object").fillna("__NA__").astype(str)
CAT_IDX = [FEATURES.index(c) for c in CAT_COLS]

# Identical construction to notebooks 01 to 03. Any change here breaks comparability
# with every row already in the ledger.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i

print(f"\n{len(train):,} train rows, {len(FEATURES)} features, cat idx {CAT_IDX}")
print(f"fold {PROBE_FOLD} holds {(folds == PROBE_FOLD).sum():,} validation rows")

In [ ]:
def run_fold(fold, iters, seed=SEED, log_every=0):
    """Train on every fold but `fold`, predict `fold`. log_every>0 prints progress."""
    tr_m, va_m = folds != fold, folds == fold
    model = cb.CatBoostClassifier(
        iterations=iters, learning_rate=LR, random_seed=seed,
        thread_count=THREADS, allow_writing_files=False, verbose=log_every,
    )
    t0 = time.time()
    model.fit(Xtr.loc[tr_m], y[tr_m], cat_features=CAT_IDX)
    secs = time.time() - t0
    p_va = model.predict_proba(Xtr.loc[va_m])[:, 1]
    return {"model": model, "p_va": p_va, "va_m": va_m, "secs": secs,
            "auc": float(roc_auc_score(y[va_m], p_va))}


def to_rank(v):
    return pd.Series(v).rank(pct=True).to_numpy()


def hhmm(secs):
    return f"{int(secs // 60)}m {int(secs % 60):02d}s"


LOG = REPO / "artifacts" / "logs" / "06_catboost_progress.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    """Print, and append to a log file flushed on every write.

    nbconvert writes the notebook only once the whole run finishes, so without
    this there is no way to watch a long run from outside the kernel. That is
    the exact blindness that killed the previous version of this notebook.
    """
    print(msg)
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{stamp}  {msg}", file=fh, flush=True)


note(f"=== run start, RUN_FULL={RUN_FULL}, threads={THREADS} ===")

## Stage 1: how long does this actually take

One fold, 200 iterations, printing every 50 so there is visible progress from the first
seconds. The only output that matters is the projection at the bottom. If it says the
full cross-validation is hours rather than tens of minutes, stop here and change the
plan rather than starting the run.

In [ ]:
bench = run_fold(PROBE_FOLD, BENCH_ITERS, log_every=50)

per_100 = bench["secs"] / BENCH_ITERS * 100
proj_probe = per_100 * FULL_ITERS / 100
proj_full = proj_probe * N_SPLITS

print(f"\n{BENCH_ITERS} iterations on one fold: {hhmm(bench['secs'])}")
print(f"fold {PROBE_FOLD} AUC at {BENCH_ITERS} iterations: {bench['auc']:.6f}")
print(f"\nrate: {per_100:.1f}s per 100 iterations\n")
print("projections, assuming the rate holds:")
print(f"  stage 2, determinism, 2 x {BENCH_ITERS} iters : {hhmm(2 * bench['secs'])}")
print(f"  stage 3, decision, 1 fold x {FULL_ITERS} iters : {hhmm(proj_probe)}")
print(f"  stage 4, full CV, {N_SPLITS} folds x {FULL_ITERS}    : {hhmm(proj_full)}")
print(f"\n  probe total, stages 1 to 3 : {hhmm(bench['secs'] * 3 + proj_probe)}")
print(f"  everything, stages 1 to 4  : "
      f"{hhmm(bench['secs'] * 3 + proj_probe + proj_full)}")
print("\nCatBoost cost is close to linear in iterations, so these are usable. They do")
print("not include prediction on the 296,302 test rows, which only stage 4 does.")

## Stage 2: is CatBoost reproducible here

LightGBM was not, and this repo found that the hard way. The cause was floating-point
gradient summation following thread scheduling, and the divergence compounded with tree
count, which is exactly why this check runs at 200 iterations rather than 2000. If it
is going to fail it fails cheaply, and 200 iterations at a fixed `thread_count` is a
fair test of the reduction order.

A failure here does not stop the notebook. It sets `DETERMINISTIC = False`, and the
verdict in stage 3 then refuses to spend the full run on a number that cannot be
repeated.

In [ ]:
a = run_fold(PROBE_FOLD, BENCH_ITERS)
b = run_fold(PROBE_FOLD, BENCH_ITERS)

d_auc = abs(a["auc"] - b["auc"])
d_row = float(np.abs(a["p_va"] - b["p_va"]).max())
DETERMINISTIC = (d_auc == 0.0) and (d_row == 0.0)

print(f"run A fold AUC : {a['auc']:.12f}   ({hhmm(a['secs'])})")
print(f"run B fold AUC : {b['auc']:.12f}   ({hhmm(b['secs'])})")
print(f"delta          : {d_auc:.3e}")
print(f"largest per-row prediction difference: {d_row:.3e}\n")

if DETERMINISTIC:
    print("bit-identical. Ledger rows from this notebook are comparable to each other.")
else:
    print("NOT DETERMINISTIC at these settings.")
    print(f"Run-to-run noise is about {d_auc:.1e} on the fold AUC, so any difference")
    print("smaller than that is meaningless and a ledger row written from this")
    print("notebook would carry it. The first thing to try is a lower THREADS.")

note(f"stage 2 done: deterministic={DETERMINISTIC}, delta={d_auc:.3e}")


## Stage 3: the decision

One fold at full iterations. That is 138,000 held-out rows, far more than a paired AUC
comparison needs, and it costs one fifth of the full cross-validation.

The comparison is **paired**: CatBoost, LightGBM, and their blend are all scored on the
same fold-0 rows. A paired difference is much more precise than either absolute AUC, so
the absolute numbers below should not be compared against five-fold ledger rows, while
the differences can be trusted.

The reference is measured on the same fold rather than quoted from `07`. The LightGBM
pair that gave the best within-family blend is scored here too, so the cross-family gain
and the within-family floor are like-for-like.

In [ ]:
probe = run_fold(PROBE_FOLD, FULL_ITERS, log_every=200)
va_m = probe["va_m"]
y_va = y[va_m]

lgb_ref = np.load(OOF_DIR / LGB_REF)[va_m]
lgb_best = np.load(OOF_DIR / LGB_BEST)[va_m]
cat_p = probe["p_va"]

auc_cat = probe["auc"]
auc_ref = float(roc_auc_score(y_va, lgb_ref))
auc_best = float(roc_auc_score(y_va, lgb_best))

print(f"\nfold {PROBE_FOLD}, {va_m.sum():,} rows, all three scored on the same rows\n")
print(f"  CatBoost {FULL_ITERS} iters : {auc_cat:.6f}   ({hhmm(probe['secs'])})")
print(f"  LightGBM exp7            : {auc_ref:.6f}")
print(f"  LightGBM exp8, best      : {auc_best:.6f}")

In [ ]:
r_cat, r_ref, r_best = to_rank(cat_p), to_rank(lgb_ref), to_rank(lgb_best)

# Cross-family blend, and the within-family blend measured on the identical rows.
cross = float(roc_auc_score(y_va, 0.5 * r_cat + 0.5 * r_best))
within = float(roc_auc_score(y_va, 0.5 * r_ref + 0.5 * r_best))

gain_cross = cross - max(auc_cat, auc_best)
gain_within = within - max(auc_ref, auc_best)
rank_corr = float(spearmanr(lgb_best, cat_p).statistic)

print(f"rank-average blends, scored on fold {PROBE_FOLD}\n")
print(f"  CatBoost + LightGBM, cross-family : {cross:.6f}   ({gain_cross:+.6f})")
print(f"  LightGBM pair, within-family      : {within:.6f}   ({gain_within:+.6f})")
print(f"\ndescriptive only, per 07: Spearman vs LightGBM exp8 = {rank_corr:.4f}")
print("  for scale, within-family pairs run 0.9741 to 0.9981 on this data, so this")
print("  number says less than it looks like it does. The blend above is the verdict.")

In [ ]:
# The bar. On a single fold the comparison is paired and therefore tighter than the
# five-fold spread, so the reference is the within-family floor measured on these same
# rows: what blending two near-identical LightGBMs is worth. A different family has to
# beat that by a clear multiple, with a floor under the bar in case the within-family
# number comes out at or below zero on this fold.
FLOOR = max(gain_within, 0.0)
BAR = max(3 * FLOOR, 0.0003)

print(f"within-family floor on this fold : {FLOOR:+.6f}")
print(f"bar for proceeding               : {BAR:+.6f}")
print(f"cross-family gain measured       : {gain_cross:+.6f}\n")

if not LEAK_OK:
    VERDICT = "blocked"
    print("VERDICT: blocked. A leak check failed at load time. Nothing below is safe.")
elif not DETERMINISTIC:
    VERDICT = "blocked"
    print("VERDICT: blocked. Stage 2 said this configuration is not reproducible, so")
    print("the difference above cannot be separated from run-to-run noise. Fix that")
    print("first. Do not spend the full run on a number that cannot be repeated.")
elif gain_cross >= BAR:
    VERDICT = "proceed"
    print("VERDICT: proceed. The cross-family blend clears the bar on held-out rows,")
    print("so CatBoost is contributing something LightGBM does not have. Set")
    print("RUN_FULL = True and Save & Run All to produce the ledger row and the")
    print("submission, then build the next family.")
elif gain_cross > 0:
    VERDICT = "marginal"
    print("VERDICT: marginal. Positive, but not clearly above what averaging two")
    print("near-identical LightGBMs already buys. A second GBDT is probably not the")
    print("lever here. Log this and go to a genuinely different family, most likely")
    print("the neural one, before spending five folds on this.")
else:
    VERDICT = "stop"
    print("VERDICT: stop. The blend does not beat the better single model on held-out")
    print("rows. More GBDT variants will not close the gap to 0.97102. Log it as a")
    print("negative result and change families.")

print(f"\nRUN_FULL is currently {RUN_FULL}.")
print("Whatever the verdict, it goes in NOTES.md. A CatBoost that fails to blend is a")
print("result worth having, because it closes off the GBDT direction cheaply.")

## Stage 4: full cross-validation

Skipped unless `RUN_FULL = True` at the top. This is the only stage that writes a ledger
row, a submission, and an out-of-fold vector, and per the repo rule that row is valid
only if this notebook was run top to bottom on a clean kernel with the flag already set.

Fold 0 is retrained rather than reused from stage 3, so all five folds come from one
identical code path. Per-fold progress and a running estimate of time left are printed,
so this can be watched instead of waited on.

In [ ]:
full = None
if not RUN_FULL:
    print("stage 4 skipped, RUN_FULL is False")
    print("the probe above is the deliverable of this run")
else:
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []
    t0 = time.time()
    for f in range(N_SPLITS):
        r = run_fold(f, FULL_ITERS, log_every=500)
        oof[r["va_m"]] = r["p_va"]
        test_pred += r["model"].predict_proba(Xte)[:, 1] / N_SPLITS
        fold_scores.append(r["auc"])
        done = time.time() - t0
        left = done / (f + 1) * (N_SPLITS - f - 1)
        note(f"stage 4 fold {f}: AUC {r['auc']:.6f} ({hhmm(r['secs'])}), "
             f"elapsed {hhmm(done)}, about {hhmm(left)} left")

    full = {"cv_mean": float(np.mean(fold_scores)),
            "cv_std": float(np.std(fold_scores)),
            "pooled": float(roc_auc_score(y, oof)),
            "secs": time.time() - t0, "oof": oof, "test_pred": test_pred}
    print(f"\nCatBoost CV {full['cv_mean']:.6f} +/- {full['cv_std']:.6f} "
          f"in {hhmm(full['secs'])}")
    print(f"pooled OOF AUC {full['pooled']:.6f}")
    print("\nThe ledger records fold-mean. pooled is printed for continuity with 07,")
    print("and the two are different statistics, so never compare across them.")

In [ ]:
if full is None:
    print("no ledger row: stage 4 did not run")
else:
    lgb_full = np.load(OOF_DIR / LGB_BEST)
    blend_oof = 0.5 * to_rank(lgb_full) + 0.5 * to_rank(full["oof"])
    per_fold = [roc_auc_score(y[folds == f], blend_oof[folds == f])
                for f in range(N_SPLITS)]
    blend_fm, blend_sd = float(np.mean(per_fold)), float(np.std(per_fold))

    print(f"5-fold rank blend with exp8: {blend_fm:.6f} +/- {blend_sd:.6f}")
    print(f"  vs best single 0.963275  : {blend_fm - 0.963275:+.6f}")
    print(f"  fold spread vs 0.000549  : {blend_sd - 0.000549:+.6f}")
    print("  within-family floor from 07 was +0.000072 with no spread reduction")

    LEDGER = REPO / "experiments.csv"
    COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
               "lb_public", "lb_private", "submitted", "notes"]
    rows = []
    if LEDGER.exists():
        with LEDGER.open(newline="", encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
    next_id = max((int(r["id"]) for r in rows), default=0) + 1

    tag = f"catboost_lr{str(LR).replace('.', '')}_n{FULL_ITERS}_seed{SEED}"
    np.save(OOF_DIR / f"{tag}.npy", full["oof"])
    sub = sample.copy()
    sub[TARGET] = full["test_pred"]
    sub.to_csv(SUB_DIR / f"{tag}.csv", index=False)

    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
    rows.append({
        "id": str(next_id), "utc": stamp, "name": "catboost_lr005",
        "cv_mean": f"{full['cv_mean']:.6f}", "cv_std": f"{full['cv_std']:.6f}",
        "folds": str(N_SPLITS), "lb_public": "", "lb_private": "", "submitted": "no",
        "notes": (f"catboost iterations={FULL_ITERS} lr={LR}, ordered target stats on "
                  f"cats, threads={THREADS}, deterministic={DETERMINISTIC}, fold-0 "
                  f"probe verdict {VERDICT}"),
    })
    rows.append({
        "id": str(next_id + 1), "utc": stamp, "name": "blend_rank_lgbm008_catboost",
        "cv_mean": f"{blend_fm:.6f}", "cv_std": f"{blend_sd:.6f}",
        "folds": str(N_SPLITS), "lb_public": "", "lb_private": "", "submitted": "no",
        "notes": (f"50/50 rank average of exp8 and experiment {next_id} OOF, "
                  f"within-family floor from 07 is +0.000072"),
    })
    with LEDGER.open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=COLUMNS)
        w.writeheader()
        w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)

    lgb_test = SUB_DIR / "lgbm_lr003_n3333_seed42.csv"
    if lgb_test.exists():
        lt = pd.read_csv(lgb_test)[TARGET].to_numpy()
        blend_sub = sample.copy()
        blend_sub[TARGET] = 0.5 * to_rank(lt) + 0.5 * to_rank(full["test_pred"])
        blend_sub.to_csv(SUB_DIR / f"blend_rank_lgbm008_{tag}.csv", index=False)
        print(f"wrote blend submission from {lgb_test.name}")
    else:
        print(f"no blend submission: {lgb_test.name} not found in submissions/")

    print(f"ledger rows {next_id} and {next_id + 1} appended")

## What this changed

**Run 2026-08-04. Verdict: stop. CatBoost is not the lever.**

Stages 2 and 3 were executed as a plain script rather than through this notebook, and
the reason is recorded in `SESSION.md`: a Jupyter kernel under a background task gets
reaped after about two minutes in this environment, while plain background python does
not. The probe writes no ledger row, so a script run is legitimate for it. Stage 4 would
still have to be a Save & Run All. The script logic is identical to the cells above.

Results on fold 0, 138,274 held-out rows, all paired on the same rows:

| | AUC |
|---|---|
| LightGBM exp8 | 0.962489 |
| LightGBM exp7 | 0.962324 |
| CatBoost, 2000 iters at lr 0.05 | 0.960814 |
| rank blend, CatBoost + exp8 | 0.962239 (-0.000250) |
| rank blend, exp7 + exp8 | 0.962519 (+0.000030) |

CatBoost is 0.001675 behind at matched budget, and blending it in is **negative** while
the within-family floor on the same rows is positive. Spearman against exp8 is 0.9877,
inside the 0.9741 to 0.9981 within-family band from `07`.

Two things this run also settled, both cheap and both worth having:

- **CatBoost is deterministic out of the box** at a fixed `thread_count`, bit-identical
  across two runs on the fold AUC and all 138,274 predictions. LightGBM needed three
  explicit flags and cost a 21-minute run to discover that it did not.
- **CatBoost is about 9x slower than LightGBM here**, 30.4s per 100 iterations, so one
  fold at 2000 iterations is 10m 10s and a five-fold run would be about 51 minutes. The
  two-hour run that was killed was this notebook's old determinism check doing two full
  five-fold passes with `verbose=0`. It was never hung.

Stage 4 was never run and should not be. The staged structure did its job: the plan was
closed off for 13 minutes of compute instead of the 102 the previous version needed to
reach the same question.